# Spectre Patch — Deep Atlas Core Builder (Colab)

Build canonical core patches for **Tile(1,1)** at substitution depths a laptop can't comfortably handle (n ≥ 8). Each output is a single `.npz` that drops into your API server's `data/atlas/` directory; the API then serves cropped requests from these cores in `O(crop)` time, never recomputing the substitution.

| Depth N | Tile count | Inscribed square (canonical) | Build wall time | Disk |
|--------:|-----------:|------------------------------:|----------------:|-----:|
| 5  |       34,649 |   146-unit square | ~5 s laptop     | 0.9 MB |
| 6  |      272,791 |   394-unit square | ~30 s laptop    | 6 MB |
| 7  |    2,147,679 |  1,033-unit square | ~5 min laptop   | 49 MB |
| **8**  | ~17 M | ~2,700-unit square (est.) | ~30–45 min Colab CPU | ~400 MB |
| **9**  | ~133 M | ~7,100-unit square (est.) | ~4–8 h Colab CPU | ~3 GB |
| **10** | ~1.05 B | ~18,000-unit square (est.) | A100 / multi-day | ~25 GB |

**To fully cover an 8192-unit-wide square** (your stated target) you need **n=10** if the mask is square-shaped. **n=9** covers a circle of diameter ≈ 14k or a hexagon of long-diagonal ≈ 14k. Build n=8 first as a smoke test, then n=9.

---

## Runtime checklist

1. **Runtime type:** `Runtime → Change runtime type`
   - **n=8**: any high-RAM CPU runtime (≥ 25 GB RAM)
   - **n=9**: A100 *isn't required* for correctness (this is CPU-bound Python) but a high-RAM runtime is mandatory; expect 50–100 GB peak RAM
   - **n=10**: only attempt if you have a long-running runtime + ≥ 200 GB free disk
2. **Persist artifacts:** mount Google Drive in Cell 2 so the build survives runtime restarts
3. **Long-running:** Colab Pro+ keeps notebooks running while you disconnect — paid runtime is recommended for n ≥ 9

## 1 — Install the `spectre_patch` package

Two ways: clone your repo or upload a wheel. Pick **one** of the cells below.

In [ ]:
# Option A — clone the public GitHub repo and install it into this Colab kernel.
# If you changed code and need a fresh pull, set FORCE_RECLONE=True.
import os, shutil, subprocess, sys

REPO_URL = os.environ.get('SPECTRE_REPO_URL', 'https://github.com/zebragum/aperiodic_monotile_API.git')
REPO_DIR = '/content/aperiodic_monotile_API'
FORCE_RECLONE = False

if FORCE_RECLONE and os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Pillow>=10.0.0', 'psutil>=5.9.0'], check=True)

# Make the editable source tree importable immediately in the current kernel,
# even if Colab's runtime path has stale state.
src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print('installed:', REPO_DIR)
print('repo url: ', REPO_URL)
print('python:   ', sys.executable)

In [ ]:
# Option B — upload spectre_patch_api as a zip via the Colab side bar (or `files.upload()`),
# unzip into /content/spectre_patch_api, then `pip install -e .`. Skip if you used Option A.
#
# from google.colab import files
# uploaded = files.upload()  # pick spectre_patch_api.zip
# !unzip -q spectre_patch_api.zip -d /content && pip install -q -e /content/spectre_patch_api
# !pip install -q Pillow>=10.0.0 psutil>=5.9.0

## 2 — Mount Google Drive (recommended for n ≥ 8)

Persist the `.npz` to Drive so a runtime disconnect doesn't lose the build. Skip this cell if you intend to download artifacts manually before disconnecting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ATLAS_OUT_DIR = '/content/drive/MyDrive/spectre_patch_atlas'
os.makedirs(ATLAS_OUT_DIR, exist_ok=True)
print('atlas dir:', ATLAS_OUT_DIR)

## 3 — Configuration

Pick the substitution depth and the inscribed-square raster resolution. Higher raster resolution gives a tighter inscribed square at proportionally more memory + time. Defaults are sized to balance the two.

In [ ]:
# === EDIT THESE ===
DEPTH = 8                # substitution iterations (8, 9, or 10)
RASTER_RES = None        # None → auto (8→2048, 9→4096, 10→8192). Override if you want.
TILE_FAMILY = 'spectre_tile_1_1'
PATCH_VERSION = '1.0.0'  # bump when you change the substitution rules

# Where the .npz lands. Override if you didn't mount Drive.
try:
    ATLAS_OUT_DIR
except NameError:
    ATLAS_OUT_DIR = '/content/atlas_out'
    import os; os.makedirs(ATLAS_OUT_DIR, exist_ok=True)

# Auto raster default — bigger raster ⇒ tighter inscribed square but more RAM.
if RASTER_RES is None:
    RASTER_RES = {7: 1024, 8: 2048, 9: 4096, 10: 8192}.get(DEPTH, 1024)

print(f'building n={DEPTH}, raster={RASTER_RES}, out={ATLAS_OUT_DIR}')

## 4 — Sanity-check imports

Confirm `spectre_patch` is importable. If this fails, redo Cell 1.

In [ ]:
# If Cell 1 was skipped or Colab lost kernel state, repair the install here.
import os, shutil, subprocess, sys, time, json, gc

REPO_URL = os.environ.get('SPECTRE_REPO_URL', 'https://github.com/zebragum/aperiodic_monotile_API.git')
REPO_DIR = '/content/aperiodic_monotile_API'

def ensure_spectre_patch_installed():
    try:
        import spectre_patch  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
    src_path = os.path.join(REPO_DIR, 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

ensure_spectre_patch_installed()

import spectre_patch
from spectre_patch import PATCH_ENGINE_SEMVER
from spectre_patch.atlas import build_core, AtlasIndex, load_core
import numpy as np

print('spectre_patch     :', spectre_patch.__file__)
print('PATCH_ENGINE_SEMVER:', PATCH_ENGINE_SEMVER)
print('numpy             :', np.__version__)
print('python            :', sys.version.split()[0])

## 5 — Build the core

This is the long-running cell. Don't interrupt it once started — the substitution recursion can't easily be resumed mid-build, and partial state isn't persisted.

What it does:
1. Builds `tile_system_after_iterations(N)` — the metatile substitution tree at depth N (RAM scales sub-linearly with N).
2. Walks every leaf of `Delta` and packs `(affine6, centroid, label_idx, dfs_path_packed, dfs_path_depth)` into numpy arrays.
3. Computes the largest fully-covered axis-aligned square (raster strategy at depth ≥ 7).
4. Writes `core_<family>_n<N>.npz` (compressed) and updates `index.json`.

In [ ]:
import psutil

p = psutil.Process()
before = p.memory_info().rss / 1e9
print(f'RSS before build: {before:.2f} GB')

t0 = time.perf_counter()
result = build_core(
    iterations=DEPTH,
    out_dir=ATLAS_OUT_DIR,
    tile_family=TILE_FAMILY,
    patch_version=PATCH_VERSION,
    overwrite=True,
    raster_resolution_override=RASTER_RES,
)
elapsed = time.perf_counter() - t0

after = p.memory_info().rss / 1e9
print()
print(f'  file:                {result.file}')
print(f'  tile_count:          {result.tile_count:,}')
print(f'  file_bytes:          {result.file_bytes:,} ({result.file_bytes/1e6:.1f} MB)')
print(f'  bbox:                {result.bbox}')
print(f'  inscribed_center:    {result.inscribed_center}')
print(f'  inscribed_half_side: {result.inscribed_half_side:.3f}')
print(f'  inscribed_method:    {result.inscribed_method}')
print(f'  builder_seconds:     {elapsed:.1f}')
print(f'  RSS after build:     {after:.2f} GB  (Δ {after-before:+.2f} GB)')

gc.collect()

## 6 — Verify the built core

Reload the freshly-written `.npz`, do a tiny probe crop, and confirm the inscribed square contains tiles at the expected density.

In [ ]:
from spectre_patch.atlas import AtlasIndex, load_core
from spectre_patch.atlas.engine import enumerate_emitted_from_core
from spectre_patch.masking import MaskSquare, RetentionMode

idx = AtlasIndex.load(ATLAS_OUT_DIR)
entry = next(e for e in idx.entries if e.iterations == DEPTH and e.tile_family == TILE_FAMILY)
core = load_core(entry, ATLAS_OUT_DIR)

print(f'loaded n={core.iterations}: {core.tile_count:,} tiles  inscribed_half_side={core.inscribed_half_side:.3f}')

# Small probe at the inscribed center.
probe_hs = min(core.inscribed_half_side * 0.05, 50.0)
mask = MaskSquare((0.0, 0.0), half_side=probe_hs)
t0 = time.perf_counter()
tiles = enumerate_emitted_from_core(
    core,
    tile_family=TILE_FAMILY,
    patch_version=PATCH_VERSION,
    seed=None,
    scale=1.0, tx=0.0, ty=0.0, rotation_deg=0.0,
    mask=mask,
    retention=RetentionMode.centroid,
)
elapsed = time.perf_counter() - t0
expected_density = (probe_hs*2)**2 / 6.93  # canonical-tile area is sqrt(3)*4 ≈ 6.93 units²
print(f'probe crop: {len(tiles):,} tiles in {(probe_hs*2):.1f}-unit square (expected ~{expected_density:.0f}) in {elapsed:.3f}s')

## 7 — Preview: rasterised inscribed square

Generate a preview PNG of the full inscribed square so you can eyeball the build before pushing it into production. SVG is impractical at 17 M tiles; rasterise to a fixed-pixel buffer instead.

*Skip this cell if you only want the `.npz`.*

In [ ]:
from spectre_patch.export.raster_export import RasterOpts, render_core_inscribed_square_png

PREVIEW_PX = 4096    # 4k preview is plenty for visual QA
preview_path = f'{ATLAS_OUT_DIR}/preview_n{DEPTH}_inscribed_{PREVIEW_PX}px.png'

opts = RasterOpts(
    pixels_per_side=PREVIEW_PX,
    background_rgb=(20, 20, 35),
    fill_rgb=(205, 214, 234),
    stroke_rgb=(23, 27, 56),
    stroke_width_px=1,
    deterministic_palette=True,
    progress_every=200_000,
)

def _on_progress(n_seen, secs):
    print(f'  ... visited {n_seen:,} tiles in {secs:.1f}s')

t0 = time.perf_counter()
info = render_core_inscribed_square_png(
    core=core,
    out_path=preview_path,
    opts=opts,
    progress=_on_progress,
)
elapsed = time.perf_counter() - t0
print()
print(f'preview written: {preview_path}')
print(f'  pixels:        {info["pixels_per_side"]} × {info["pixels_per_side"]}')
print(f'  tiles_drawn:   {info["tiles_drawn"]:,}')
print(f'  out_bytes:     {info["out_bytes"]:,} ({info["out_bytes"]/1e6:.1f} MB)')
print(f'  elapsed:       {elapsed:.1f}s')

from IPython.display import Image
Image(filename=preview_path)

## 8 — Package + download

If you mounted Drive, the `.npz` is already persisted. Otherwise download the archive now (before the runtime disconnects).

In [ ]:
import shutil, os, glob

core_file = result.file  # full path to core_*_n{DEPTH}.npz
index_file = os.path.join(ATLAS_OUT_DIR, 'index.json')

archive_root = '/content/spectre_atlas_artifact'
os.makedirs(archive_root, exist_ok=True)
shutil.copy2(core_file, archive_root)
shutil.copy2(index_file, archive_root)
for png_path in glob.glob(f'{ATLAS_OUT_DIR}/preview_n{DEPTH}_*.png'):
    shutil.copy2(png_path, archive_root)

archive_name = f'spectre_atlas_n{DEPTH}'
archive = shutil.make_archive(f'/content/{archive_name}', 'zip', archive_root)
print('archive:', archive, f'({os.path.getsize(archive)/1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(archive)
except Exception:
    print('not in Colab; archive lives at', archive)

## 9 — Deploy into the API

On your API host:

```bash
# 1) drop the .npz into the atlas dir referenced by SPECTRE_PATCH_ATLAS_DIR
scp spectre_atlas_n8.zip user@host:/srv/spectre/
ssh user@host
cd /srv/spectre
unzip spectre_atlas_n8.zip -d data/atlas/

# 2) verify the manifest now sees the new core
python -m spectre_patch.atlas.cli list --out data/atlas

# 3) restart the API so the lifespan reload picks it up
systemctl restart spectre-patch-api

# 4) sanity
curl http://localhost:8000/v1/capabilities | jq '.atlas.cores[] | select(.iterations==8)'
```

The new core now serves any request whose mask half-side falls inside its inscribed square.

## Troubleshooting

**OOM during enumeration** — the depth-N substitution tree itself is small, but the *leaf array* is `T × 64 bytes ≈ 1 GB at n=8`, `8 GB at n=9`. If you hit OOM, switch to a high-RAM runtime; there is no chunked builder yet.

**`raster_…` inscribed-square method takes hours** — the raster grid is `RASTER_RES²` cells; halve `RASTER_RES` to trade tighter inscribed-square accuracy for runtime.

**Wandering inscribed center across depths** — expected. The atlas selector ignores center alignment and the engine transparently shifts mask coordinates onto each core's inscribed center. See `src/spectre_patch/atlas/engine.py`.

**ID stability** — the `.npz` stores DFS-path bit-packed indices, so `stable_tile_id()` round-trips between substitution and atlas modes for the same `(family, patch_version, seed, depth, path)`. Bumping `patch_version` is the only way to invalidate IDs.